# ASDEA_sensors -- full tour

Extensive reference of **every method with all its inputs**. The package is
currently a skeleton (methods raise `NotImplementedError`), so this notebook is
an **API reference**: the cells show the intended calls and parameters, not a
run yet. As each layer is implemented, the matching cells become runnable.

Conventions:
- Units are SI: acceleration `m/s^2`, velocity `m/s`, displacement `m`.
- `ds.MOF00135.<method>` runs on one sensor; `ds.<method>` broadcasts to all.
- `component="x"|"y"|"z"` returns one result; `"all"` returns a dict per axis.
- Results are cached on the dataset and invalidated automatically.

## 0. Setup

In [ ]:
import sys
sys.path.insert(0, "../src")   # run the notebook from examples/

from asdea_sensors import SensorDataset
from asdea_sensors.config import settings

DATA = r"C:\\Users\\ppala\\Desktop\\02_31MAY2026"   # folder with the .h5 files

## 1. Instantiate and inspect

In [ ]:
ds = SensorDataset(
    path         = DATA,
    pattern      = "*.h5",        # which files to index
    date_source  = "filename",    # "filename" (YYYYMMDDHHMMSS) | "timestamp"
    devices      = None,          # None = all; or ["MOF00135", "MOF00134"]
    load_mode    = "auto",        # "auto" | "ram" | "lazy"
    ram_fraction = 0.5,           # switch to lazy above this fraction of free RAM
    axes_map     = None,          # None = config.SENSOR_AXES
    verbose      = True,          # print the summary block
)

In [ ]:
ds.n_jobs   = 4         # internal batch engine
ds.parallel = True

ds.devices              # list of device ids
ds.fs                   # sampling frequency (Hz), from Timestamp
ds.dt                   # sampling interval (s)
ds.time_span            # (first_datetime, last_datetime)
ds.duration             # human-readable span
ds.axes_map             # per-sensor axis mapping
ds.summary()            # reprint the summary block
ds.cache_summary()      # what is cached and how much RAM it uses

In [ ]:
# Sensor geometry (floors known; UTM/azimuth to be filled in)
settings.SENSOR_GEOMETRY
settings.SENSOR_AXES        # the non-standard orientation mapping

## 2. Per-sensor access, chaining and broadcast

In [ ]:
dev = ds.MOF00135               # DeviceHandle (same as ds.device("MOF00135"))

# per sensor
ds.MOF00135.newmark(component="x")

# broadcast to every device -> dict keyed by device
ds.newmark(component="x")

# component="all" -> dict per axis
ds.MOF00135.peaks(component="all")

## 3. Windows (start + length, or explicit bounds)

In [ ]:
w = ds.MOF00135.window(start="2026-05-31 15:00:00", length="60min")   # "300sec", "2hour", seconds
w = ds.MOF00135.window(start="2026-05-31 18:30:00", length="300sec")
sub = ds.MOF00135.get_window(t0="2026-05-31 15:00", t1="2026-05-31 16:00")

## 4. Resample (dataset, sensor, signal)

In [ ]:
ds2  = ds.resample(dt=0.01)             # whole dataset to dt=0.01 s
ds2  = ds.resample(fs=100)              # or by target fs
dev2 = ds.MOF00135.resample(dt=0.005)   # one sensor

## 5. Signal pipeline -- decoupled steps

In [ ]:
sig = ds.MOF00135.signal(
    components  = "all",      # "x" | "y" | "z" | "all"
    mode        = "auto",     # "ram" | "lazy" | "auto"
    workers     = 4,          # parallel file reads
    remove_mean = False,
)

sig = sig.baseline(method="polynomial", components="all")      # step 1: correct acc only
sig = sig.filter(fmin=0.1, fmax=24.9, engine="obspy",          # step 2: bandpass only
                 order=4, zerophase=True, components="all")
sig = sig.derive(method="trapezoid", remove_mean=True,         # step 3: integrate to vel/disp
                 components="all")

sig.acc_x, sig.vel_x, sig.disp_x
sig.time, sig.dt, sig.fs, sig.n, sig.duration, sig.axes

## 6. Seismic

### Newmark (full output)

In [ ]:
spec = ds.MOF00135.newmark(
    component  = "x",      # "x" | "y" | "z" | "all"
    source     = "acc",
    zeta       = 0.05,     # damping ratio
    max_period = 5.01,     # s
    dT         = 0.01,     # period step (s)
    factor     = 1.0,      # 1/9.81 to present PSa/Sa in g (PSv, Sd unchanged)
)
spec["T"]
spec["PSa"], spec["PSv"]
spec["Sd"], spec["Sv"], spec["Sa"]
spec["u"], spec["v"], spec["a"], spec["at"]

### RotD (selectable percentile, matrix cached)

In [ ]:
r50  = ds.MOF00135.rotd(comp_x="x", comp_y="y", rotd=50, damping=0.05,
                        angle_step=5, max_period=5.01, dT=0.01)
r100 = ds.MOF00135.rotd(rotd=100)     # reuses the cached PSa matrix
r50["ROTD50"], r50["angle_rotd50"], r50["PSa_matrix"]
r50["PSa_geo_mean"], r50["PSa_arith_mean"], r50["PSa_SRSS"]

### Arias, CAV, Housner, Peaks

In [ ]:
arias = ds.MOF00135.arias(component="x", low=5, high=95)
arias["IA_percent"], arias["t_start"], arias["t_end"], arias["IA_total"], arias["pot_dest"]

cav  = ds.MOF00135.cav(component="x")                       # cav["CAV"], cav["curve"]
hous = ds.MOF00135.housner(component="x", T1=0.1, T2=2.5, zeta=0.05)   # hous["SI"]
pk   = ds.MOF00135.peaks(component="all")                   # PGA / PGV / PGD per axis

### Fourier, PSD, STFT

In [ ]:
fou = ds.MOF00135.fourier(component="x", num_frequencies=4, prominence=1e-6,
                          distance_frac=0.02, smooth=None, bexp=40)
fou["freqs"], fou["spectrum"], fou["dom_freqs"], fou["dom_periods"], fou["dom_peaks"]

psd = ds.MOF00135.psd(component="x", nperseg=256, noverlap=128, window="hann",
                      bands=[(0,1),(1,2.5),(2.5,5),(5,10)], detrend="constant")
psd["f"], psd["Pxx"], psd["band_energy"]

stft = ds.MOF00135.stft(component="x", nperseg=256, noverlap=224, window="hann", fmax=25.0)
stft["f"], stft["t"], stft["Zxx"]

## 7. Building (multi-sensor, uses geometry)

In [ ]:
# Transfer function: one pair, or stacked base -> each floor of the array
frf  = ds.transfer_function(numerator="MOF00135", denominator="MOF00134", component="x",
                            estimator="H1", nperseg=1024, noverlap=512, window="hann",
                            smooth="konno", bexp=40, fmax=25.0)
frfs = ds.transfer_function(base="MOF00134", floors="all", component="x")

# Coherence: one pair, or the full matrix
coh  = ds.coherence("MOF00135", "MOF00134", component="x", nperseg=1024, noverlap=512)
cohm = ds.coherence_matrix(component="x")

# Modal: frequencies, shapes, damping, tracking
freqs  = ds.modal_frequencies(component="x")
shapes = ds.mode_shapes(component="x")                       # amplitude/phase per floor
modal  = ds.MOF00135.modal_tracking(component="x", window="10min", overlap=0.5,
                                    fband=(1.0, 8.0), n_modes=2, smooth="konno", bexp=40)

# Torsion of floor 4 (pair MOF00135 + MOF00136)
tor = ds.torsion(floor=4, component="x")

# Drift profile over the vertical array, interstory drift, base rocking
prof  = ds.drift_profile(component="x")
drift = ds.interstory_drift(upper="MOF00135", lower="MOF00134", component="x")
rock  = ds.base_rocking()

## 8. Ambient (step by step, no filter inside)

In [ ]:
config = {
    "Fs": ds.fs, "STA": 1.0, "LTA": 30.0,
    "vent": 60.0, "vmin": 0.2, "vmax": 2.5,
    "p": 0.05, "bexp": 40, "f1": 0.1, "f2": 25.0,
    "vent_seismic": False,
}

sig = ds.MOF00135.window("2026-05-31 15:00", "30min").signal().baseline().filter(0.1, 24.9)
amb = sig.ambient(config, component="x")

amb.sta_lta()                       # 1
amb.select_windows(manual=None)     # 2
amb.taper()                         # 3
amb.fft(apply_filter=False)         # 4
amb.smooth()                        # 5
amb.average()                       # 6
amb.sta_lta_ratio, amb.windows_signal, amb.win_ids, amb.mean_spectrum, amb.dominant_period

# HVSR (Nakamura) and amplification
hv  = ds.MOF00135.hvsr(config=config, comp_h=("x","y"), comp_v="z", combine="geometric")
amp = ds.amplification(ref="MOF00135", others=["MOF00134","MNAT0031"],
                       basis="fourier", component="x", config=config)

## 9. Batch (broadcast and time sweep)

In [ ]:
all_newmark = ds.newmark(component="x")             # parallel over all devices
sweep = ds.sweep(devices="all", length="60min", overlap=0.0,
                 analyses=["newmark", "arias", "ambient"], component="x", config=config)

## 10. Export to .h5 with Provenance (replicable)

In [ ]:
ds.export_h5("run_31MAY.h5")                        # all sensors + analyses + global provenance
ds.MOF00135.export_h5("MOF00135_31MAY.h5")          # one sensor (device chain)

from asdea_sensors.io.results_file import ResultsFile
r = ResultsFile("run_31MAY.h5")
r.provenance                                        # input files, pipeline, params, config + hash
r.analyses("MOF00135")
data, params = r.get("MOF00135", "newmark_x")

## 11. Plotting (decoupled, reads the cache)

In [ ]:
ds.MOF00135.plot_signals(components="all", kind="acc", save=None)
ds.MOF00135.plot_newmark(component="x", quantity="PSa")
ds.MOF00135.plot_rotd(rotd=[0, 50, 100])
ds.MOF00135.plot_fourier(component="x", smooth="konno")
ds.MOF00135.plot_stft(component="x")
ds.MOF00135.plot_psd(component="x")
ds.MOF00135.plot_arias(component="x")
ds.transfer_function_plot(numerator="MOF00135", denominator="MOF00134", component="x")
ds.MOF00135.plot_modal_tracking(component="x")
ds.amplification_plot(ref="MOF00135", others=["MOF00134"], basis="fourier")